In [ ]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool

from com.example.agentic.agent.LLMManager import LLMManager
llm = LLMManager.create_llm('openai')
llm.call('Hello')

In [ ]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage

# Create custom storage with specific embedder
# custom_storage = KnowledgeStorage(
#     embedder={
#         "provider": "ollama",
#         "config": {"model": "mxbai-embed-large"}
#     },
#     collection_name="my_custom_knowledge"
# )

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict


class ScanPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    summary: str = Field(description="The key findings or insights about this topic")
    benefits: List[Dict[str, str]] = Field(
        description="The key benifit and details for topic",
        default_factory=list
    )
    references: List[str] = Field(
        description="the key reference design diagram url for discussed topic",
        default_factory=list
    )

In [ ]:
from crewai_tools import FileReadTool

# Initialize the tool to read any files the agents knows or lean the path for
file_read_tool = FileReadTool()

# OR

# Initialize the tool with a specific file path, so the agent can only read the content of the specified file
file_read_tool = FileReadTool(file_path='path/to/your/file.txt')

In [ ]:
from com.example.agentic.tools.config import _embedder_config_ollama,_embedder_config_openai

# Agent city expert
scan_doc_agent = Agent(
    role="As a software architect with expertise in {topic}",
    goal="Design robust, scalable system architectures that balance performance, maintainability, and cost-effectiveness",
    backstory="With 20+ years of experience building {topic} systems at scale, you've developed a pragmatic approach"
              "to software architecture. You've seen both successful and failed systems and have learned valuable lessons from each." 
              "You balance theoretical best practices with practical constraints and always consider the maintenance and operational" 
              "aspects of your designs."
    tools=[],
    verbose=True,
    knowledge_sources=[text_kw_source],
    embedder=_embedder_config_openai,
    llm=llm,
    allow_delegation=False,
)


# Define Tasks
scan_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="outputs/L006/scan_doc-summary_{run_id}.json"
)


# Review the context you got and expands each topic into full section for a report about {topic}
# Make sure you find top 10 interesting and relevant design url about {topic} and return list of url
# Define Tasks
format_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="outputs/L006/format_doc-summary_{run_id}.json" 
)

In [ ]:

# Assemble the Crew
from crewai import Crew, Process
from datetime import datetime

_exe_date = datetime.now().strftime('%Y-%m-%d')
_exe_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Crew triggered on {_exe_date} with execution id {_exe_id}")

crew = Crew(
    agents=[scan_doc_agent],
    tasks=[scan_doc_task],
    process=Process.sequential,
    share_crew=False,
    verbose=True
)

inputs = {
    'topic': 'Microservice Design',
    'date': _exe_date,
    'run_id': _exe_id
}

# Execute the Tasks
result = crew.kickoff(inputs=inputs)
print(result)

In [ ]:
from crewai_tools import RagTool
from com.example.agentic.tools.config import _tool_config,_rag_tool_config

# Create a RAG tool with custom configuration
rag_tool = RagTool(config=_rag_tool_config, summarize=True)



In [ ]:
rag_tool.add(file_path='/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt')

In [ ]:
r = rag_tool.run("JPMORGAN")
r

In [ ]:
from github import Github, Auth
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

github = GitHubAPIWrapper()
toolkit = GitHubToolkit.from_github_api_wrapper(github)

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

# Use the new Auth class to handle your token
auth = Auth.Token(_git_hub_token)

# Authenticate using a Personal Access Token
github = Github(auth=auth)

# Get a specific repository
repo = github.get_repo(_repo)

print(repo.stargazers_count)

In [ ]:
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

github = GitHubAPIWrapper(
    active_branch="main", 
    github_repository="brijeshdhaker/docker-hadoop-cluster",
    
)
# The wrapper automatically looks for GITHUB_PERSONAL_ACCESS_TOKEN if not passed
toolkit = GitHubToolkit.from_github_api_wrapper(github)
tools = toolkit.get_tools()
for tool in tools:
    print(tool.name)

In [ ]:
from langchain_community.document_loaders import GithubFileLoader

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

loader = GithubFileLoader(
    repo=_repo,  # the repo name
    branch="main",  # the branch name
    access_token=_git_hub_token,
    github_api_url="https://api.github.com",
    file_filter=lambda file_path: file_path.endswith(
        ".md"
    ),  # load all markdowns files.
)
documents = loader.load()

In [ ]:
documents[1].page_content

In [ ]:
from crewai_tools import GithubSearchTool

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

# Initialize the tool with your PAT
github_tool = GithubSearchTool(
    config=_rag_tool_config,
    github_repo='https://github.com/brijeshdhaker/docker-hadoop-cluster',
    gh_token=_git_hub_token,
    content_types=['code', 'issue']
)

# Use the tool within an Agent
# from crewai import Agent

# github_specialist = Agent(
#     role='GitHub Researcher',
#     goal='Search for specific code patterns and issues in the repository',
#     backstory='Expert in navigating complex codebases and tracking development issues.',
#     tools=[github_tool],
#     verbose=True
# )




In [ ]:
github_tool.add(repo='brijeshdhaker/docker-hadoop-cluster',content_types=['code', 'issue'])
github_tool.run('Search for specific code patterns and issues in the repository')